## Setup — run this first

Mounts Drive, points the notebook at your project folder, and installs
what's missing. No git, no tokens.

**Your Drive folder must look like this:**

```
MyDrive/Ghana_Dropout_Project_R02/
├── config.py          <- these three at the TOP level,
├── losses.py             not inside notebooks/
├── pipeline.py
├── requirements.txt
├── notebooks/         <- the 11 notebooks
└── data-raw/
    └── ghana_dropout_study_M.xlsx
```

`results/`, `figures/`, `models/` and `data-processed/` are created for you.

Drive saves as it goes, so there is nothing to push — but see the checklist
in the last cell before you submit.


In [13]:
# ============================================================
# SETUP — Google Drive. Run first. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

PROJECT = "/content/drive/MyDrive/Ghana_Dropout_Project_R02"   # <-- edit if yours differs
RAW_XLSX_NAME = "ghana_dropout_study_M.xlsx"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

    root = Path(PROJECT)
    if not root.exists():
        raise FileNotFoundError(
            f"{PROJECT} does not exist.\n"
            "Create that folder in My Drive and put config.py, losses.py, "
            "pipeline.py, requirements.txt, the notebooks/ folder and "
            "data-raw/ inside it."
        )

    # the three modules must sit at the project root, not in notebooks/
    missing = [m for m in ("config.py", "losses.py", "pipeline.py")
               if not (root / m).exists()]
    if missing:
        stray = [m for m in missing if (root / "notebooks" / m).exists()]
        msg = f"Missing from {PROJECT}: {missing}"
        if stray:
            msg += (f"\n{stray} are in notebooks/ instead. Move them UP one "
                    "level, into the project folder itself. If they stay in "
                    "notebooks/, that folder gets treated as the project root "
                    "and results/ is written in the wrong place.")
        raise FileNotFoundError(msg)

    os.chdir(root)
    os.environ["DROPOUT_REPO"] = str(root)

    # Forget any previously loaded copy of the project modules. Python keeps
    # the first version it imported for the whole session, so an edited
    # config.py is silently ignored until the runtime restarts. This makes
    # every run use the files currently in Drive.
    for _m in ("config", "losses", "pipeline"):
        sys.modules.pop(_m, None)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    # ---- dependencies: only install what is actually missing ------------
    need = []
    for mod, pkg in [("lightgbm", "lightgbm"), ("shap", "shap"),
                     ("catboost", "catboost"), ("xgboost", "xgboost"),
                     ("imblearn", "imbalanced-learn"), ("openpyxl", "openpyxl")]:
        try:
            __import__(mod)
        except ImportError:
            need.append(pkg)
    if need:
        print("installing:", need)
        subprocess.run(f"pip install -q {' '.join(need)}", shell=True)
    else:
        print("all dependencies present")

    # ---- raw workbook ---------------------------------------------------
    (root / "data-raw").mkdir(exist_ok=True)
    xlsx = root / "data-raw" / RAW_XLSX_NAME
    if xlsx.exists():
        print(f"raw workbook: {xlsx.name}")
    else:
        loose = list(root.glob(RAW_XLSX_NAME)) + list(root.glob(f"**/{RAW_XLSX_NAME}"))
        if loose:
            import shutil
            shutil.copy(loose[0], xlsx)
            print(f"copied {loose[0]} -> data-raw/")
        else:
            print(f"NOT FOUND: data-raw/{RAW_XLSX_NAME}\n"
                  "Notebook 1 needs it. Notebooks 2-9 read "
                  "data-processed/cleaned_data.csv instead and are fine "
                  "without it.")

    print(f"\nPROJECT : {os.getcwd()}")
else:
    print("Not in Colab — paths resolve from the project root.")


all dependencies present
raw workbook: ghana_dropout_study_M.xlsx

PROJECT : /content/drive/MyDrive/Ghana_Dropout_Project_R02


# Notebook 4 — Baseline Classifiers

## What changed from R01

| Change | Reason |
|---|---|
| `cross_validate` no longer runs on the full `X, y` | R01 cell 15 cross-validated over all 1000 rows, so the test partition was inside the CV training folds. All CV is now on the training pool |
| **No test-set scoring at all here** | GATE-1(iii) — the same seed-42 partition was scored in Notebooks 4, 5b, 6, 6b, 7 and 8. The test set is now scored once, in Notebook 8 |
| Preprocessing via `preprocess_inside_fold` | Same pipeline as every other notebook, so the six-model comparison is commensurable with the focal-loss experiment |
| Primary metric is **AUC-PR**, ranking on AUC-PR | R01 ranked and selected on F1 while M15 declared AUC-PR primary |
| Per-fold scores committed | Q8 — needed for paired tests and variance |
| Equal budget stated explicitly, zero trials each | GATE-2 disclosure |

Output feeds **Supplementary Table S1**, which objective 2 promises and R01
never supplied. The data existed in `baseline_results.csv` all along, so
supplying it is the cheap route to closing half that objective.

In [14]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      score_binary, two_level_variance, raw_feature_cols)

banner("NOTEBOOK 4 — BASELINES")
OUT = run_dir("notebook04_baselines")
capture_environment(OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)
print(f"train_pool {train_pool.shape}  |  test NOT touched in this notebook")

BASELINE_SEEDS = SEEDS[:3]     # 3 seeds x 25 folds x 6 models; widen if you want
print(f"seeds: {BASELINE_SEEDS}, folds per seed: {N_SPLITS*N_REPEATS}")

NOTEBOOK 4 — BASELINES
repo            : /content/drive/MyDrive/Ghana_Dropout_Project_R02
provenance      : NONE — set FREEZE_TAG in config.py before scoring the test set
school_handling : drop
FEATURE SET     : records_plus_questionnaire   (SECONDARY — includes friend-reported questionnaire items)
primary metric  : auc_pr
train_pool (784, 42)  |  test NOT touched in this notebook
seeds: [42, 123, 456], folds per seed: 25


In [16]:
# ---- model pool ---------------------------------------------------------
# All six receive: the identical in-fold pipeline, the identical folds, and
# ZERO hyperparameter search trials. That is parity at the floor, and it is
# stated rather than implied (GATE-2).
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def make_models(seed):
    m = {
        "Logistic Regression": LogisticRegression(
            random_state=seed, max_iter=2000, class_weight="balanced"),
        "Decision Tree": DecisionTreeClassifier(
            random_state=seed, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(
            random_state=seed, n_estimators=200, class_weight="balanced"),
    }
    try:
        from xgboost import XGBClassifier
        m["XGBoost"] = XGBClassifier(random_state=seed, eval_metric="logloss",
                                     verbosity=0)
    except ImportError:
        print("  xgboost unavailable — skipped (state this in M11)")
    try:
        from lightgbm import LGBMClassifier
        # SHARED_PARAMS already carries verbosity; passing it again is a
        # duplicate-keyword TypeError.
        m["LightGBM"] = LGBMClassifier(objective="binary", random_state=seed,
                                       **SHARED_PARAMS)
    except ImportError:
        print("  lightgbm unavailable — skipped")
    try:
        from catboost import CatBoostClassifier
        m["CatBoost"] = CatBoostClassifier(random_state=seed, verbose=0,
                                           allow_writing_files=False)
    except ImportError:
        print("  catboost unavailable — skipped (state this in M11)")
    return m

NEEDS_SCALING = {"Logistic Regression"}
print("models:", list(make_models(42)))

models: ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost', 'LightGBM', 'CatBoost']


In [17]:
# ---- run: in-fold preprocessing, training pool only --------------------
import time
rows = []
t0 = time.perf_counter()
for seed in BASELINE_SEEDS:
    for fi, (tr, vl) in enumerate(cv_splits(train_pool, seed), 1):
        X_tr, y_tr, X_vl, y_vl, meta = preprocess_inside_fold(
            train_pool.iloc[tr], train_pool.iloc[vl])
        for name, model in make_models(seed).items():
            A, B = X_tr, X_vl
            if name in NEEDS_SCALING:
                sc = StandardScaler().fit(X_tr)          # fit on train fold only
                A = pd.DataFrame(sc.transform(X_tr), columns=X_tr.columns)
                B = pd.DataFrame(sc.transform(X_vl), columns=X_vl.columns)
            t = time.perf_counter()
            model.fit(A, y_tr)
            fit_s = time.perf_counter() - t
            p = model.predict_proba(B)[:, 1]
            rows.append({"seed": seed, "fold": fi, "arm": name,
                         "search_trials": 0, "n_features": X_tr.shape[1],
                         "fit_seconds": fit_s, **score_binary(y_vl, p)})
    print(f"  seed {seed} done [{time.perf_counter()-t0:.0f}s]")

fold_df = pd.DataFrame(rows)
fold_df.to_csv(OUT / "baseline_fold_scores.csv", index=False)
print(f"\n{len(fold_df)} fold-level rows")

  seed 42 done [137s]
  seed 123 done [262s]
  seed 456 done [386s]

450 fold-level rows


In [18]:
# ---- Supplementary Table S1 (the table objective 2 promised) -----------
var = two_level_variance(fold_df, PRIMARY_METRIC)

agg = (fold_df.groupby("arm")
       .agg(auc_pr_mean=("auc_pr", "mean"),
            auc_roc_mean=("auc_roc", "mean"),
            recall_mean=("recall", "mean"),
            precision_mean=("precision", "mean"),
            macro_f1_mean=("macro_f1", "mean"),
            accuracy_mean=("accuracy", "mean"),
            mean_fit_seconds=("fit_seconds", "mean"))
       .reset_index()
       .merge(var[["arm", "between_seed_sd", "mean_within_seed_fold_sd"]], on="arm")
       .sort_values("auc_pr_mean", ascending=False))
agg["search_trials"] = 0
agg["n_seeds"] = len(BASELINE_SEEDS)
agg["n_folds_per_seed"] = N_SPLITS * N_REPEATS
agg["base_rate_pct"] = round(100 * float(train_pool[TARGET].mean()), 1)
agg["n_positive_train_pool"] = int(train_pool[TARGET].sum())
agg.to_csv(OUT / "supplementary_table_S1.csv", index=False)

print("SUPPLEMENTARY TABLE S1 — six-classifier comparison")
print("(cross-validated on the training pool; NO test-set figures)\n")
print(agg.drop(columns=["accuracy_mean"]).round(4).to_string(index=False))
print(f"\nRanked on {PRIMARY_METRIC}, matching M15. R01 ranked on F1.")
print(f"Base rate {agg['base_rate_pct'].iloc[0]}% on "
      f"{agg['n_positive_train_pool'].iloc[0]} positive cases — print both "
      "beside every figure in this table (J4).")

plt.figure(figsize=(9, 4.5))
o = agg.sort_values("auc_pr_mean")
plt.barh(o["arm"], o["auc_pr_mean"],
         xerr=o["between_seed_sd"], color="steelblue")
plt.axvline(float(train_pool[TARGET].mean()), ls="--", c="r", lw=1,
            label="base rate (uninformative)")
plt.xlabel("AUC-PR (mean over seeds, error bars = between-seed SD)")
plt.legend(); plt.tight_layout()
plt.savefig(OUT / "figures/baseline_comparison.png", dpi=200); plt.close()

SUPPLEMENTARY TABLE S1 — six-classifier comparison
(cross-validated on the training pool; NO test-set figures)

                arm  auc_pr_mean  auc_roc_mean  recall_mean  precision_mean  macro_f1_mean  mean_fit_seconds  between_seed_sd  mean_within_seed_fold_sd  search_trials  n_seeds  n_folds_per_seed  base_rate_pct  n_positive_train_pool
      Random Forest       0.9930        0.9992       0.8956          0.9844         0.9649            0.5280           0.0010                    0.0121              0        3                25            8.8                     69
           LightGBM       0.9884        0.9971       0.9160          0.9696         0.9671            0.2076           0.0037                    0.0192              0        3                25            8.8                     69
           CatBoost       0.9882        0.9984       0.9091          0.9805         0.9681            3.7530           0.0021                    0.0176              0        3                2

In [19]:
# ---- substrate justification (M11) -------------------------------------
print("VARIANCE, BOTH LEVELS (J3 — separately AND comparably):")
print(var.round(4).to_string(index=False))

best = agg.iloc[0]
print(f"\nstrongest baseline on {PRIMARY_METRIC}: {best['arm']} "
      f"({best['auc_pr_mean']:.4f})")
if "LightGBM" in set(agg["arm"]):
    lg = agg[agg["arm"] == "LightGBM"].iloc[0]
    print(f"LightGBM: {lg['auc_pr_mean']:.4f} "
          f"(gap to best: {best['auc_pr_mean']-lg['auc_pr_mean']:+.4f}, "
          f"between-seed SD {lg['between_seed_sd']:.4f})")
    print("\nM11's justification stands on two legs and should say both: "
          "LightGBM is competitive with the strongest baseline (quantified "
          "above), AND it is the only one of the six that accepts a fully "
          "custom training objective — which the intervention requires.")

write_manifest(OUT, {"notebook": "04_baselines",
                     "test_set_scored": False,
                     "seeds": BASELINE_SEEDS,
                     "models": list(agg["arm"]),
                     "search_trials_per_model": 0,
                     "ranking_metric": PRIMARY_METRIC})
print("\nNEXT: Notebook 5 (class imbalance, resampling inside the fold).")

VARIANCE, BOTH LEVELS (J3 — separately AND comparably):
                arm   mean  between_seed_sd  mean_within_seed_fold_sd  n_seeds  n_folds_per_seed label
      Random Forest 0.9930           0.0010                    0.0121        3                25   NaN
           LightGBM 0.9884           0.0037                    0.0192        3                25   NaN
           CatBoost 0.9882           0.0021                    0.0176        3                25   NaN
            XGBoost 0.9818           0.0019                    0.0262        3                25   NaN
Logistic Regression 0.9551           0.0031                    0.0410        3                25   NaN
      Decision Tree 0.8205           0.0233                    0.1108        3                25   NaN

strongest baseline on auc_pr: Random Forest (0.9930)
LightGBM: 0.9884 (gap to best: +0.0047, between-seed SD 0.0037)

M11's justification stands on two legs and should say both: LightGBM is competitive with the strongest b

---

## Before you submit

Drive has already saved everything — nothing to push. But two things still
have to happen before submission, and neither is automatic.


In [20]:
# ---- what this run produced, and what is still owed ----
import os, sys
from pathlib import Path

try:
    latest = sorted(Path(OUT).parent.glob("*"))[-1]
    files = sorted(p.relative_to(OUT).as_posix() for p in Path(OUT).rglob("*")
                   if p.is_file())
    print(f"run directory : {Path(OUT).relative_to(REPO)}")
    print(f"files written : {len(files)}")
    for f in files:
        print("   ", f)
except Exception as e:
    print("no run directory recorded in this session:", e)

print("""
────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,
                   requirements.txt, README.md, and all of results/

2. SET config.FREEZE_TAG BEFORE SCORING THE TEST SET.
   Without git there is no commit hash to anchor the freeze to. Put a
   fixed dated string in config.py — e.g. "R02-freeze-2026-09-25-1430" —
   at the moment you freeze the configuration, and never revise it.
   Notebook 8 refuses to score the test set until it is set.
────────────────────────────────────────────────────────────────────""")


run directory : results/notebook04_baselines/20260922T223218Z_records_plus_questionnaire
files written : 6
    RUN_MANIFEST.json
    baseline_fold_scores.csv
    environment_versions.csv
    figures/baseline_comparison.png
    pip_freeze.txt
    supplementary_table_S1.csv

────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,
                   requi